# 07 – Parameter estimates: PINN vs Mechanistic

Comparison of the posteriors estimated by the two models, week by week, with **split violins**: for every week the left half is the PINN and the right half the Mechanistic model, with the median as a short line. Only the weeks that both models have are shown.

One notebook for the four synthetic scenarios and for the real case, chosen with `CASE` (as in notebooks 03–06):

| `CASE` | Parameters that can be plotted | Reference lines |
|---|---|---|
| `"synthetic"` (R0 = 1.5 / 2.0, noise `"light"` / `"mild"`) | `"R0"` and/or `"Rt"` (effective reproduction number) | true R0 (R0 plot), Rt = 1, week of the peak |
| `"real"` | `"Rt"` only | Rt = 1, week of the peak |

The synthetic scenarios show the weeks 4-30 (R0 = 1.5) and 4-20 (R0 = 2.0). For them there is also the **density of R0 at the last week** (week 34 for R0 = 1.5, week 24 for R0 = 2.0), PINN vs Mechanistic, with the true value.

Input: `posteriors/pinn/<case>/<run>/mcmc_distributions.pkl` (notebook 03; `<run>` is `best_WIS` or `seed_<seed>`, see `PINN_SEED`) and `posteriors/mechanistic/<case>/abc_distributions.pkl` (notebook 05), from `precomputed/` or `outputs/`. Output: `figures/parameters/<case>/`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from pinnvsmech.evaluation.best_seed import resolve_run_dir
from pinnvsmech.inference.cases import load_case
from pinnvsmech.plotting.params import (
    DENSITY_XLIM, default_peak_week, default_t0_range, load_distributions, load_mech_last_week_R0, load_pinn_R0,
    parameter_spec, plot_params_comparison_violin, plot_R0_density,
)
from pinnvsmech.paths import FIGURES_DIR, OUTPUTS_DIR, PRECOMPUTED_DIR

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 1.5                      # synthetic only: 1.5 or 2.0
NOISE = "light"                    # synthetic only: "light" or "mild"
CASE_OVERRIDES = dict(dt_train=None)   # look-back on all the history, as the Mechanistic "nall" results (it is
                                   # the default of the synthetic scenarios; for the real case the default is 5 weeks)

PARAMETERS = ["Rt"]                # "R0" (synthetic only) and/or "Rt": one figure for each
# PARAMETERS = ["R0", "Rt"]

SOURCE = "precomputed"             # "precomputed" (results of the paper) or "outputs" (new runs)
PINN_SEED = "best"                 # PINN run: "best" = folder best_WIS of the case, or the seed chosen in notebook 10; or the seed number (e.g. 10) to use seed_010

T0_MIN, T0_MAX = None, None        # first / last week shown (index); None: default of the case (see below)
PEAK_WEEK = "default"              # "default": synthetic = peak of the observations, real = (6.55, 7.45); None: not shown;
                                   # or a band (a, b)
DENSITY = True                     # synthetic only: density of R0 at the last week (week 34 / 24)
LABEL_PINN, LABEL_MECH = "PINN", "Mechanistic"

SAVE_FIGURES = False
FIGURE_FORMAT = "pdf"              # "pdf" or "png"

## Data

In [ ]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
# The Mechanistic results do not depend on the weights of the likelihood
mech_label = load_case(CASE, R0=R0_TRUE, noise=NOISE,
                       **{k: v for k, v in CASE_OVERRIDES.items() if k != "weight"}).label

root = (PRECOMPUTED_DIR if SOURCE == "precomputed" else OUTPUTS_DIR) / "posteriors"
pinn_run = resolve_run_dir(PINN_SEED, root / "pinn" / case.label)
data_pinn = load_distributions(pinn_run / "mcmc_distributions.pkl")
data_mech = load_distributions(root / "mechanistic" / mech_label / "abc_distributions.pkl")
print(case.label, f"| PINN run {pinn_run.name} | PINN weeks:", data_pinn["weeks"][[0, -1]], "| Mechanistic weeks:", data_mech["weeks"][[0, -1]])

# Weeks shown and week of the peak
t0_min, t0_max = default_t0_range(CASE, case.dates, R0=R0_TRUE)
t0_min = T0_MIN if T0_MIN is not None else t0_min
t0_max = T0_MAX if T0_MAX is not None else t0_max
peak_week = default_peak_week(CASE, case.obs) if PEAK_WEEK == "default" else PEAK_WEEK
print("weeks shown:", t0_min, "to", t0_max, "| peak band:", peak_week)

## Figures

In [ ]:
if CASE == "synthetic":
    side_label, figsize, true_values = rf"$\mathbf{{R_0={R0_TRUE}}}$", (10, 3.6), {"R0s_samples": R0_TRUE}
else:
    side_label, figsize, true_values = "COVID-19\nLondon", (10, 4.5), None

fig_dir = FIGURES_DIR / "parameters" / case.label
os.makedirs(fig_dir, exist_ok=True)

for which in PARAMETERS:
    spec = parameter_spec(which, CASE, R0=R0_TRUE)        # R0 is refused for the real case
    fig = plot_params_comparison_violin(
        data_pinn, data_mech, dates=case.dates, label_a=LABEL_PINN, label_b=LABEL_MECH,
        params=[spec], peak_week=peak_week, t0_min=t0_min, t0_max=t0_max, true_values=true_values,
        side_label=side_label, show_title=False, figsize=figsize,
    )
    if SAVE_FIGURES:
        fig.savefig(fig_dir / f"{which}_violin.{FIGURE_FORMAT}", dpi=300, bbox_inches="tight")
    plt.show()

## Density of R0 at the last week (synthetic scenarios)

In [ ]:
if CASE == "synthetic" and DENSITY:
    r0_pinn = load_pinn_R0(data_pinn, case.t0_last)                 # estimation at t0 = 33 (R0 = 1.5) / 23 (R0 = 2.0)
    r0_mech = load_mech_last_week_R0(data_mech, case.t0_last)
    for name, samples in (("PINN", r0_pinn), ("Mechanistic", r0_mech)):
        print(name, "median", round(float(np.median(samples)), 3), "5-95%", np.percentile(samples, [5, 95]).round(3))

    fig, ax = plot_R0_density(
        {"PINN": r0_pinn, "Mechanistic": r0_mech}, true_value=R0_TRUE, show_legend=False, ylabel="Scaled density",
        figsize=(6, 4), xlim=DENSITY_XLIM[float(R0_TRUE)],
        margins=dict(left=0.17, right=0.97, bottom=0.19, top=0.91),
    )
    if SAVE_FIGURES:
        fig.savefig(fig_dir / f"R0_density_week{case.t0_last + 1}.{FIGURE_FORMAT}", dpi=300)
    plt.show()